# Attention

Example:

> Buffalo buffalo Buffalo buffalo buffalo buffalo Buffalo buffalo.

The same token, `buffalo`, can mean different things depending on where it appears.

## 1. Goal

We want:

```text
sequence
↓
contextualized sequence
```

In other words, we want to turn each token's **initial representation**:

```text
x₁, x₂, x₃, ...
```

into a representation that incorporates context:

```text
h₁, h₂, h₃, ...
```

A useful intuition:

> Ambiguity lives in the initial representation; context resolves it.

---

## 2. Why do we need an initial representation?

If every token started with the same representation:

```text
I        → [0, 0, 0]
love     → [0, 0, 0]
buffalo  → [0, 0, 0]
pizza    → [0, 0, 0]
```

then:

```text
I love pizza
```

and:

```text
buffalo buffalo buffalo
```

would look identical to the model.

So contextualization must start from an **informative initial state**.

---

## 3. Why not one-hot?

One-hot vectors are already enough to distinguish token identity.

In fact:

```text
embedding lookup = one-hot × embedding matrix
```

So the question is not:

> Can we distinguish different tokens?

The real question is:

> What kind of starting representation is useful for later computation?

With one-hot vectors, all different tokens are:

```text
equally unrelated
```

An embedding instead gives each token a **learned initial representation**.

---

## 4. The real problem

Now we have:

```text
token
↓
initial embedding
```

The real question becomes:

> How do we turn a sequence of initial representations into contextualized representations?

Think of it as an API:

```python
contextualize(sequence) -> contextualized_sequence
```

---

## 5. Dumbest implementation: Giant MLP

Forget attention for now.

We could simply define:

```text
max_length = 1024
```

Pad shorter sequences and include positional information.

Then do:

```text
1024 embeddings
↓
flatten
↓
giant MLP
↓
1024 contextualized representations
```

In principle:

> This is a perfectly valid implementation of `contextualize()`.

So attention does not need to answer:

> "Can an MLP do this?"

It needs to answer:

> **Can we design a better contextualization operator than a Giant MLP?**

That is the real starting point for attention.

## 6. Why Token-to-Token Interaction?

Question:
> How can one token read another token and change its own representation?

Can MLP do this? Yes, UAT shows it can.

But for token A to read from B, it's implicit and undirect.

Why not
> Let A and B directly perform a computation.

Then
> What computation?

---

## 7. Start with Cosine Similarity

Can we use cosine score?
> More similar → more useful.

### Good
- sequence-level func → token-to-token
- same rule can compare any pair of tokens
- scalar result

### Problems

`cos(A, B) = cos(B, A)`

But we want:

`score(A, B) != score(B, A)`

We needs an **asymmetric** relationship:

> A → B

"What is A looking for, and does B have it?"

> B → A

"What is B looking for, and does A have it?"

The result should be different!

---

## 8. Query and Key

To make:
`score(A, B) != score(B, A)`

each token needs 2 representations instead of just themselves. Ex. Query and Key.

Then:

`Query_A · Key_B`

can differ from:

`Query_B · Key_A`

Intuitively:

> Query = what am I looking for?

> Key = what do I have?

Cosine similarity measures how aligned two vectors are. dot product is already the core of that idea.

So we can use their dot product as the relevance score. 


### Summarize

For token i attending to token j:

`score(i, j) = q_i * k_j`

In practice, we compute all query-key pairs at once:

`scores = QK^T`

---

## 9. Why Value?

If A decides to listen to B:

> What does A actually receive from B?
> 
> Why not just return B?

Say you go to see a doctor. Do you tell the doctor everything from your life?

No!

For this task, **every** patients will fill out the same **form** that organize their info so that the result suits the doctor's needs.

Patient info -> embedding
form -> W_V

> Isn't filling out form will lose info?

In real life, yes.

In matrix multiplication, it depends. If input dimension == output dimension and full rank mul, no info lost. If output dimension is less than input, there is info lost.

But it's ok. Doctor doesn't care who your first love is 😂.

> And then there is residual connection!
x' = x + Attention(x)

So original info has a path forward.
